In [1]:
pip install chemotools


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import spectral
import cv2
import numpy as np
import matplotlib.pyplot as plt

caminho =  r"C:\Users\Gabriela\Documents\GitHub\IC_2025\Ic2025\data_processed\ATCC13_240506-161053.hdr"

img = spectral.open_image(caminho).load()
rgb = spectral.get_rgb(img,[50,30, 10])

gray = cv2.cvtColor((rgb *255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
gray = cv2.GaussianBlur(gray,(7,7),2)

circles = cv2.HoughCircles(gray,cv2.HOUGH_GRADIENT,1.2,100,param1=100,param2=30,minRadius=50,maxRadius=200)
mask = np.zeros(gray.shape,dtype=np.uint8)
x,y,r= circles[0][0]
cv2.circle(mask,(int(x),int(y)),int(r),1,thickness=-1)

mask3d = np.repeat(mask[:,:,np.newaxis],img.shape[2],axis=2)
roi = img * mask3d

roi_rgb = spectral.get_rgb(roi, [50, 30, 10])
roi_rgb_norm = roi_rgb / roi_rgb.max()
#pegar o roi e colocar em 0 e 1 depois aplicar o pre processamento  

altura, largura, n_bandas = img.shape
img_array = img 

n_amostras = altura * largura

# Transformar a imagem em matriz 2D (pixels x bandas)
espectros_matriz = img.reshape(n_amostras, n_bandas)

espectros_matriz_norm = (espectros_matriz - espectros_matriz.min(axis=1, keepdims=True)) / \
                        (espectros_matriz.max(axis=1, keepdims=True) - espectros_matriz.min(axis=1, keepdims=True) + 1e-8)

c:\Users\Gabriela\Documents\GitHub\IC_2025\.venv\Lib\site-packages\spectral\io\envi.py:187: UserWarning: Parameters with non-lowercase names encountered and converted to lowercase. To retain source file parameter name capitalization, set spectral.settings.envi_support_nonlowercase_params to True.
  warnings.warn(msg)
C:\Users\Gabriela\AppData\Local\Temp\ipykernel_15160\3379238669.py:20: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  roi = img * mask3d


In [1]:
import spectral
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

caminho =  r"C:\Users\Gabriela\Documents\GitHub\IC_2025\Ic2025\data_processed\ATCC13_240506-161053.hdr"

img = spectral.open_image(caminho).load()
rgb = spectral.get_rgb(img,[50,30, 10])

gray = cv2.cvtColor((rgb *255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
gray = cv2.GaussianBlur(gray,(7,7),2)

circles = cv2.HoughCircles(gray,cv2.HOUGH_GRADIENT,1.2,100,param1=100,param2=30,minRadius=50,maxRadius=200)
mask = np.zeros(gray.shape,dtype=np.uint8)
x,y,r= circles[0][0]
cv2.circle(mask,(int(x),int(y)),int(r),1,thickness=-1)

mask3d = np.repeat(mask[:,:,np.newaxis],img.shape[2],axis=2)
roi = img * mask3d

roi_rgb = spectral.get_rgb(roi, [50, 30, 10])
roi_rgb_norm = roi_rgb / roi_rgb.max()

altura, largura, n_bandas = roi.shape

roi_matriz = roi.reshape(altura * largura, n_bandas)

roi_matriz = roi_matriz[np.any(roi_matriz != 0, axis=1)]

roi_matriz_std = (roi_matriz - roi_matriz.mean(axis=1, keepdims=True)) / \
                 (roi_matriz.std(axis=1, keepdims=True) + 1e-8)

df_roi = pd.DataFrame(roi_matriz_std)

# (Opcional) adicionar nomes de colunas representando as bandas espectrais
df_roi.columns = [f"banda_{i}" for i in range(df_roi.shape[1])]

# Caminho de saída do CSV
caminho_saida = r"C:\Users\Gabriela\Documents\GitHub\IC_2025\Ic2025\notebooks\roi_padronizado.csv"

# Salvar em CSV
df_roi.to_csv(caminho_saida, index=False)

c:\Users\Gabriela\Documents\GitHub\IC_2025\.venv\Lib\site-packages\spectral\io\envi.py:187: UserWarning: Parameters with non-lowercase names encountered and converted to lowercase. To retain source file parameter name capitalization, set spectral.settings.envi_support_nonlowercase_params to True.
  warnings.warn(msg)
C:\Users\Gabriela\AppData\Local\Temp\ipykernel_10856\3906597429.py:21: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  roi = img * mask3d


In [3]:
import pandas as pd

caminho_entrada =r"C:\Users\Gabriela\Documents\GitHub\IC_2025\Ic2025\notebooks\roi_padronizado.csv"
df = pd.read_csv(caminho_entrada)
roi_padronizado = df.values

In [4]:
from chemotools.scatter import StandardNormalVariate

snv = StandardNormalVariate() 
spectra_snv = snv.fit_transform(roi_padronizado)

df_snv = pd.DataFrame(spectra_snv, columns=df.columns)  

caminho_saida = r"C:\Users\Gabriela\Documents\GitHub\IC_2025\Ic2025\notebooks\roi_snv.csv"

# Salvar em CSV
df_snv.to_csv(caminho_saida, index=False)